In [ ]:
import os
from llama_index.core import VectorStoreIndex, Document, SimpleDirectoryReader, PromptTemplate
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from langchain_openai import ChatOpenAI

In [10]:
template = (
    "You're provided with the context information delimited by triple brackets. \n"
    "You're also provided with the users query delimited by triple dashes. \n"
    "If you don't find the information in the context information, provide the answer from your prior knowledge. Tell me if this information isn't provided in the context and that you're using your prior knowledge. \n"
    ""

         "{context_str}\n"
    "''' \n"
    "''' \n"
         "{query_str}\n"
    "''' \n"
)
qa_template = PromptTemplate(template)

In [5]:
os.environ["OPENAI_API_KEY"] = "REMOVED_OPENAI_KEY"

In [ ]:
# Configure settings
#Settings.llm = ChatOpenAI(model = "gpt-4.1-2025-04-14", temperature = 0.1)
Settings.llm = OpenAI(model = "gpt-4.1-mini", temperature = 0.1)
Settings.embed_model = OpenAIEmbedding(model = "text-embedding-3-large")

#upload the documents (mention your own directory)
document = SimpleDirectoryReader("/Users/adhvik_rayaprolu/Downloads/random").load_data()

# Create vector index (training phase)
index = VectorStoreIndex.from_documents(document)

# Create query engine
query_engine = index.as_query_engine(similarity_top_k = 3, text_qa_template = qa_template)

# Query and return response
response = query_engine.query("How many players are on a pitch in football?") # provided by llm 
# whether we want llm to answer from its own knowledge or not, for this we need to use prompt template

print(response)

The information about the number of players on a pitch in football is not provided in the context you shared. Using my prior knowledge: In association football (soccer), there are 22 players on the pitch at the start of a match—11 players for each team.


In [15]:
#store the index in our drive or any other storage location
from llama_index.core import StorageContext, load_index_from_storage

In [16]:
model_storage_directory = "/Users/adhvik_rayaprolu/Downloads/test_index"

In [17]:
index.storage_context.persist(model_storage_directory)

In [18]:
# loading the stored index
storage_context = StorageContext.from_defaults(persist_dir = model_storage_directory)

Loading llama_index.core.storage.kvstore.simple_kvstore from /Users/adhvik_rayaprolu/Downloads/test_index/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from /Users/adhvik_rayaprolu/Downloads/test_index/index_store.json.


In [19]:
loaded_index = load_index_from_storage(storage_context)

In [20]:
query_engine_new = loaded_index.as_query_engine()

In [22]:
response = query_engine_new.query("What is RAG?")
print(response)

RAG, or Retrieval-Augmented Generation, is a technique that enhances language models by incorporating your own data into their responses. It works by loading and preparing your data for queries, indexing it so it can be efficiently searched, and then using user queries to filter down to the most relevant information. This relevant context, along with the user's query, is sent to the language model, which generates a response. RAG is commonly used in applications like query engines, chatbots, and agents to ensure that responses are informed by both the model's training data and your specific data.
